### ЗАДАЧА: Сверка платежей между сайтом и платежным шлюзом

Финансовая команда получила две выгрузки за день:
- журнал заказов из сайта,
- журнал транзакций из платежного шлюза.

Нужно выполнить сверку, чтобы понять:
- какие платежи подтверждены корректно,
- где есть расхождения по сумме,
- какие заказы остались без успешной оплаты,
- какие транзакции пришли без соответствующего заказа.

После обработки нужно сохранить:
- публичный отчет в `JSON` для финансовой аналитики,
- полный внутренний снимок в `pickle` для аудита.

НЕОБХОДИМО:

1. Преобразовать `order_rows` в список словарей `orders`.
2. Преобразовать `payment_rows` в список словарей `payments`.
3. Для всех денежных полей привести тип `float`.
4. Построить словарь `payments_by_order_id`, где ключ — `order_id`.
5. Для каждого заказа определить статус сверки:
   - `matched`, если успешный платеж найден и сумма совпадает,
   - `amount_mismatch`, если успешный платеж найден, но сумма отличается,
   - `missing_payment`, если успешного платежа нет.
6. Собрать список `reconciliation_results` с полями:
   - `order_id`
   - `customer`
   - `expected_amount`
   - `paid_amount`
   - `status`
7. Собрать список `orphan_payments` — успешные платежи, у которых нет заказа.
8. Построить словарь `status_summary` по статусам сверки с количеством записей.
9. Посчитать:
   - `expected_total`
   - `matched_total`
   - `mismatch_total`
10. Отсортировать `reconciliation_results` по `status`, а затем по `order_id`.
11. Собрать `public_report` со структурой:
   - `report_name`
   - `status_summary`
   - `expected_total`
   - `matched_total`
   - `mismatch_total`
   - `reconciliation_results`
   - `orphan_payments`
12. Сохранить `public_report` в `payment_reconciliation_report.json`.
13. Сохранить полный снимок в `payment_reconciliation_snapshot.pkl`.
14. Прочитать оба файла обратно и вывести результаты.


In [1]:
import json
import pickle

# order row format: order_id|customer|channel|amount|currency
order_rows = [
    "ORD-501|Anna|site|1250.00|RUB",
    "ORD-502|Ivan|site|890.50|RUB",
    "ORD-503|Olga|mobile|3490.00|RUB",
    "ORD-504|Max|site|999.99|RUB",
    "ORD-505|Lena|mobile|1540.00|RUB",
]

# payment row format: transaction_id|order_id|paid_amount|currency|gateway_status
payment_rows = [
    "TX-9001|ORD-501|1250.00|RUB|success",
    "TX-9002|ORD-502|800.50|RUB|success",
    "TX-9003|ORD-503|3490.00|RUB|failed",
    "TX-9004|ORD-504|999.99|RUB|success",
    "TX-9005|ORD-999|4200.00|RUB|success",
    "TX-9006|ORD-505|1540.00|RUB|success",
]

orders = []
payments = []
payments_by_order_id = {}
reconciliation_results = []
orphan_payments = []
status_summary = {}

for row in order_rows:
    order_id, customer, channel, amount_raw, currency = row.split("|")

    # TODO 1: преобразуйте amount_raw в float и сохраните в amount
    amount = float(amount_raw)

    order = {
        "order_id": order_id,
        "customer": customer,
        "channel": channel,
        # TODO 2: добавьте amount
        "amount": amount,
        "currency": currency,
    }
    orders.append(order)

for row in payment_rows:
    transaction_id, order_id, paid_amount_raw, currency, gateway_status = row.split("|")

    # TODO 3: преобразуйте paid_amount_raw в float и сохраните в paid_amount
    paid_amount = float(paid_amount_raw)

    payment = {
        "transaction_id": transaction_id,
        "order_id": order_id,
        # TODO 4: добавьте paid_amount
        "paid_amount": paid_amount,
        "currency": currency,
        "gateway_status": gateway_status,
    }
    payments.append(payment)

    # TODO 5: если gateway_status == 'success', сохраните payment в payments_by_order_id по ключу order_id
    if gateway_status == 'success':
        payments_by_order_id[order_id] = payment

expected_total = 0
matched_total = 0
mismatch_total = 0

for order in orders:
    # TODO 6: увеличьте expected_total на order['amount']
    expected_total += order['amount']

    payment = payments_by_order_id.get(order["order_id"])

    # TODO 7: если успешный платеж не найден, статус = 'missing_payment' и paid_amount = 0.0
    if payment is None:
        status = 'missing_payment'
        paid_amount = 0.0
    else:
        # TODO 8: если платеж найден и сумма совпадает, статус = 'matched'
        if payment['paid_amount'] == order['amount']:
            status = 'matched'
            paid_amount = payment['paid_amount']
        # TODO 9: если платеж найден, но сумма не совпадает, статус = 'amount_mismatch'
        else:
            status = 'amount_mismatch'
            paid_amount = payment['paid_amount']

    # TODO 10: если статус == 'matched', увеличьте matched_total
    if status == 'matched':
        matched_total += paid_amount
    # TODO 11: если статус == 'amount_mismatch', увеличьте mismatch_total на paid_amount
    elif status == 'amount_mismatch':
        mismatch_total += paid_amount

    result = {
        "order_id": order["order_id"],
        "customer": order["customer"],
        "expected_amount": order["amount"],
        # TODO 12: добавьте paid_amount
        "paid_amount": paid_amount,
        # TODO 13: добавьте status
        "status": status
    }
    reconciliation_results.append(result)

    # TODO 14: увеличьте счетчик status_summary для текущего status
    if status in status_summary:
        status_summary[status] += 1
    else:
        status_summary[status] = 1

order_ids = set()

# TODO 15: заполните set order_ids всеми order_id из orders
for order in orders:
    order_ids.add(order['order_id'])

for payment in payments:
    # TODO 16: если payment успешный и его order_id отсутствует в order_ids,
    # добавьте в orphan_payments словарь с полями:
    #   transaction_id, order_id, paid_amount, currency
    if payment['gateway_status'] == 'success' and payment['order_id'] not in order_ids:
        orphan_payment = {
            "transaction_id": payment['transaction_id'],
            "order_id": payment['order_id'],
            "paid_amount": payment['paid_amount'],
            "currency": payment['currency']
        }
        orphan_payments.append(orphan_payment)

# TODO 17: отсортируйте reconciliation_results по status и order_id
reconciliation_results.sort(key=lambda x: (x['status'], x['order_id']))

public_report = {
    "report_name": "daily_payment_reconciliation",
    "status_summary": status_summary,
    # TODO 18: добавьте expected_total
    "expected_total": expected_total,
    # TODO 19: добавьте matched_total
    "matched_total": matched_total,
    # TODO 20: добавьте mismatch_total
    "mismatch_total": mismatch_total,
    # TODO 21: добавьте reconciliation_results
    "reconciliation_results": reconciliation_results,
    # TODO 22: добавьте orphan_payments
    "orphan_payments": orphan_payments
}

snapshot = {
    "orders": orders,
    "payments": payments,
    "payments_by_order_id": payments_by_order_id,
    "reconciliation_results": reconciliation_results,
    "orphan_payments": orphan_payments,
    "status_summary": status_summary,
}

# TODO 23: сохраните public_report в payment_reconciliation_report.json через json.dump
#   используйте ensure_ascii=False и indent=2
with open('payment_reconciliation_report.json', 'w', encoding='utf-8') as f:
    json.dump(public_report, f, ensure_ascii=False, indent=2)

# TODO 24: сохраните snapshot в payment_reconciliation_snapshot.pkl через pickle.dump
with open('payment_reconciliation_snapshot.pkl', 'wb') as f:
    pickle.dump(snapshot, f)

# TODO 25: прочитайте payment_reconciliation_report.json в loaded_report через json.load
with open('payment_reconciliation_report.json', 'r', encoding='utf-8') as f:
    loaded_report = json.load(f)

# TODO 26: прочитайте payment_reconciliation_snapshot.pkl в loaded_snapshot через pickle.load
with open('payment_reconciliation_snapshot.pkl', 'rb') as f:
    loaded_snapshot = pickle.load(f)

print("Заказы:")
print(orders)
print()

print("Платежи:")
print(payments)
print()

print("Результаты сверки:")
print(reconciliation_results)
print()

print("Отчет:")
print(public_report)
print()

print("Данные из JSON:")
print(loaded_report)
print()

print("Данные из pickle:")
print(loaded_snapshot)


Заказы:
[{'order_id': 'ORD-501', 'customer': 'Anna', 'channel': 'site', 'amount': 1250.0, 'currency': 'RUB'}, {'order_id': 'ORD-502', 'customer': 'Ivan', 'channel': 'site', 'amount': 890.5, 'currency': 'RUB'}, {'order_id': 'ORD-503', 'customer': 'Olga', 'channel': 'mobile', 'amount': 3490.0, 'currency': 'RUB'}, {'order_id': 'ORD-504', 'customer': 'Max', 'channel': 'site', 'amount': 999.99, 'currency': 'RUB'}, {'order_id': 'ORD-505', 'customer': 'Lena', 'channel': 'mobile', 'amount': 1540.0, 'currency': 'RUB'}]

Платежи:
[{'transaction_id': 'TX-9001', 'order_id': 'ORD-501', 'paid_amount': 1250.0, 'currency': 'RUB', 'gateway_status': 'success'}, {'transaction_id': 'TX-9002', 'order_id': 'ORD-502', 'paid_amount': 800.5, 'currency': 'RUB', 'gateway_status': 'success'}, {'transaction_id': 'TX-9003', 'order_id': 'ORD-503', 'paid_amount': 3490.0, 'currency': 'RUB', 'gateway_status': 'failed'}, {'transaction_id': 'TX-9004', 'order_id': 'ORD-504', 'paid_amount': 999.99, 'currency': 'RUB', 'gate